# Project 1: SKU Assessment and Attention Ranking

**FMN Inventory Attention Engine**

This stage creates the canonical SKU assessment consumed by the future Streamlit UI and grounded AI layer. Risk and ranking remain deterministic.

## Objective

Translate forecast and inventory risk into a planner facing queue that answers: **What should I investigate first, and why?**

In [ ]:
from pathlib import Path
import pandas as pd
import sys
ROOT = Path('/mnt/data/fmn_project1')
sys.path.insert(0, str(ROOT))
from src.assess import rank_attention, attention_summary

assessments = pd.read_csv(ROOT / 'artifacts/evaluation/sku_assessments.csv')
summary = attention_summary(assessments)
summary

### Finding

The canonical assessment now covers **all 28 SKUs**, including the three new SKUs that were intentionally excluded from the established SKU LightGBM path and SKU 1017, which required a transparent available history fallback.

In [ ]:
cols = ['attention_rank','sku_id','sku_type','risk_state','attention_priority',
        'current_stock','lead_time_days','planning_horizon_days',
        'lead_time_range_days','coverage_days','days_to_projected_stockout',
        'projected_unmet_units','recommendation']
assessments[cols].head(20)

### Finding

The queue is exception first. Critical SKUs receive P1, Overstock receives P3, and Healthy SKUs remain available for investigation without competing with exceptions.

In [ ]:
attention = assessments[assessments['risk_state'] != 'Healthy'].copy()
print(f'Attention queue: {len(attention)} of {len(assessments)} SKUs')
print(attention[['attention_rank','sku_id','risk_state','attention_priority','attention_score']].to_string(index=False))

### Finding

The current as of 29 June 2026 queue contains **10 Critical** and **6 Overstock** SKUs. The three new SKUs are Critical because they have zero stock, limited history, and inconsistent observed lead times.

## Canonical contract

The `SkuAssessment` contract carries the evidence needed by both the UI and AI layer: stock, forecast, lead time, planning horizon, delivery information, projected stock, risk state, drivers, data quality flags, and recommendation.

The LLM will consume this contract. It will not calculate risk or invent evidence.

In [ ]:
sample = assessments.loc[assessments['sku_id'].isin(['SKU-2000','SKU-1014','SKU-1012']),
                         ['sku_id','risk_state','planning_horizon_days','lead_time_range_days','drivers','recommendation']]
sample

### Finding

New SKUs are explicitly marked through `sku_type`, a 14 day planning horizon, and a lead time range rather than pretending that their unstable lead time is a reliable point estimate.

## Important implementation correction

The first risk output contained 24 SKUs because the established LightGBM feature path requires 28 days of complete history. That was not acceptable for the product contract because the assessment contains 28 SKUs. We corrected the handoff rather than silently dropping the four excluded SKUs.

* SKU 1017: transparent 28 day available history fallback because one unresolved demand value prevents the complete LightGBM feature row.
* SKUs 2000–2002: cold start using their 12 day observed history and a 14 day planning horizon, with uncertainty and lead time inconsistency surfaced.

## Decision

The analytical product core now has a single contract for all SKUs. We are ready to build the **Attention Center and SKU Drilldown UI** against this contract. The AI explanation layer should be added after the UI can already function without an LLM.